In [1]:
import scipy.io 
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.neighbors import LocalOutlierFactor
import time

We import the dataset and convert the .mat format to a pandas dataframe.

In [3]:
satellite = scipy.io.loadmat('satellite.mat')
shuttle = scipy.io.loadmat('shuttle.mat')

X1 = satellite['X']
y1 = satellite['y']

satellite_df = pd.DataFrame(X1)
satellite_df['y'] = y1.ravel()

X2 = shuttle['X']
y2 = shuttle['y']

shuttle_df = pd.DataFrame(X2)
shuttle_df['y'] = y2.ravel()

#satellite_df
#shuttle_df


Satellite has 6435 instances and 36 attributes, shuttle has 49097 instances and 9 attributes. Both have a predictor y, which will be used to evaluate on later. This is consistent with the paper. 

We define the isolation forest first.

In [6]:
def isolation_forest(X, t=100, s=256):
    """
    Returns the fit, prediction and anomaly scores of an isolation forest, using the input dataset X, the number of trees t and the subsampling size s.
    """
    iforest = IsolationForest(
        n_estimators=t,
        max_samples=s,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    )

    iforest_fit = iforest.fit(X)
    iforest_prediction = iforest_fit.predict(X)
    iforest_prediction = (iforest_prediction == -1).astype(int)
    iforest_scores = iforest_fit.score_samples(X)

    return iforest_fit, iforest_prediction, iforest_scores


isolation_forest(satellite_df.drop(columns=["y"]))

(IsolationForest(max_samples=256, n_jobs=-1, random_state=42),
 array([1, 0, 0, ..., 0, 0, 0], shape=(6435,)),
 array([-0.51038056, -0.43054116, -0.40541205, ..., -0.42976863,
        -0.44679514, -0.46535485], shape=(6435,)))

In [7]:
def pruning(D, alpha, m, iforest_scores):
    """
    Prune the (1 - theta_D) fraction with the lowest
    anomaly scores and return the candidate outlier set.
    """

    # Dispersion coefficients
    means = D.mean(axis=0)
    stds = D.std(axis=0, ddof=0)

    outlier_coef = stds / means.abs().replace(0, np.nan)
    outlier_coef = outlier_coef.replace(
        [np.inf, -np.inf], np.nan
    )

    valid_coef = outlier_coef.dropna()

    m = max(1, min(int(m), len(valid_coef)))
    top_m = valid_coef.nlargest(m)

    # Trim threshold
    theta_D = alpha * top_m.sum() / m
    theta_D = float(np.clip(theta_D, 0, 1))

    # Ordering of scores
    if isinstance(iforest_scores, pd.Series):
        scores = iforest_scores.reindex(D.index)

    else:
        scores = pd.Series(
            np.asarray(iforest_scores),
            index=D.index
        )

    if len(scores) != len(D):
        raise ValueError(
            "iforest_scores must have the same length as D."
        )

    # Pruning step
    n_prune = int(np.floor((1 - theta_D) * len(D)))

    prune_indices = scores.nlargest(n_prune).index

    D_candidates = D.drop(index=prune_indices)

    return D_candidates, prune_indices, theta_D, outlier_coef

We will finally calculate the lof for each value in the candidate set

In [9]:
def calculate_lof_score(D_candidates, lofk, m_lof):
    """
    Calculates the lof score based on the LocalOutlierFactor. Outputs the expected outliers, the top_m indices of outliers, 
    the lof values, the lof object and the prediction vector.
    """
    lof = LocalOutlierFactor(
        n_neighbors=lofk,
        contamination="auto"
    )

    lof_prediction = lof.fit_predict(D_candidates)
    lof_prediction = (lof_prediction == -1).astype(int)
    lof_scores = -lof.negative_outlier_factor_
    
    lof_indices = np.argsort(lof_scores)[::-1]
    
    m_lof = min(m_lof, len(D_candidates))
    top_m_indices = lof_indices[:m_lof]

    # Retrieve the top m observations and their scores
    D_outliers = D_candidates.iloc[top_m_indices].copy()
    D_outliers["lof_score"] = lof_scores[top_m_indices]

    return D_outliers, top_m_indices, lof, lof_prediction, lof_scores

We chain these together to form our IF-LOF function.

In [11]:
def IF_LOF(X, t=100, s=256, alpha = 1, top_m = 10, lofk = 5, m_lof = 100): 
    """
    Combines the isolation forst, pruning and the final lof prediction in one function. Returns a prediction of outliers.
    """
    iforest_scores = isolation_forest(X, t, s)[2]
    D_candidates = pruning(X, alpha, top_m, iforest_scores)[0]
    D_outliers = calculate_lof_score(D_candidates, lofk, m_lof)[0]

    lof_prediction = pd.Series(0, index=X.index, dtype=int)
    lof_prediction.loc[D_outliers.index] = 1
    
    return lof_prediction

For our evaluation, we define a function to compare the predictions with the real values and output the accuracy and F-measure.

In [13]:
def evaluation(y_true, y_pred):
    """
    Based on prediction and true value outputs the accuracy and F-measure.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel().tolist()
    acc = (tp + tn) / (tn + fp + fn + tp)
    f_measure = (2*tp) / (2*tp + fp + fn)
    return acc, f_measure

We also define a function that keeps track of the running time.

In [15]:
def time_execution(func, *args, **kwargs):
    """
    Tracks running time of a function. Outputs the function and running time.
    """
    start = time.perf_counter()
    result = func(*args, **kwargs)
    end = time.perf_counter()

    elapsed = end - start
    return result, elapsed

For the satellite and the shuttle dataset, we make our predictions for the functions and record the time needed.

In [46]:
# Satellite
y_true_sat = satellite_df['y'].to_numpy()
X_sat = satellite_df.drop(columns=["y"])
t, s, alpha, top_m, lofk, m_lof = 200, 256, 1, 10, 10 , 2036 

y_iforest_pred_sat, time_if_sat = time_execution(
    isolation_forest, X_sat, t, s
)
y_iforest_pred_sat = y_iforest_pred_sat[1]

y_lof_pred_sat, time_lof_sat = time_execution(
    calculate_lof_score, X_sat, lofk, m_lof
)
y_lof_pred_sat = y_lof_pred_sat[3]

y_if_lof_pred_sat, time_if_lof_sat = time_execution(
    IF_LOF, X_sat, t, s, alpha, top_m, lofk, m_lof
)

# Shuttle
y_true_shut = shuttle_df['y'].to_numpy()
X_shut = shuttle_df.drop(columns=["y"])
t, s, alpha, top_m, lofk, m_lof = 200, 256, 1, 5, 10 , 3511 

y_iforest_pred_shut, time_if_shut = time_execution(
    isolation_forest, X_shut, t, s
)
y_iforest_pred_shut = y_iforest_pred_shut[1]

y_lof_pred_shut, time_lof_shut = time_execution(
    calculate_lof_score, X_shut, lofk, m_lof
)
y_lof_pred_shut = y_lof_pred_shut[3]

y_if_lof_pred_shut, time_if_lof_shut = time_execution(
    IF_LOF, X_shut, t, s, alpha, top_m, lofk, m_lof
)

We will finally print the results. Per dataset we compare IF, LOF and IF-LOF on accuracy, F-measure and run time.

In [48]:
print('Satellite; (accuracy, F-measure)')
print('IF:', evaluation(y_true_sat, y_iforest_pred_sat),
      f'| Runtime: {time_if_sat:.4f} seconds')
print('LOF:', evaluation(y_true_sat, y_lof_pred_sat),
      f'| Runtime: {time_lof_sat:.4f} seconds')
print('IF-LOF:', evaluation(y_true_sat, y_if_lof_pred_sat),
      f'| Runtime: {time_if_lof_sat:.4f} seconds')

print('\nShuttle; (accuracy, F-measure)' )
print('IF:', evaluation(y_true_shut, y_iforest_pred_shut),
      f'| Runtime: {time_if_shut:.4f} seconds')
print('LOF:', evaluation(y_true_shut, y_lof_pred_shut),
      f'| Runtime: {time_lof_shut:.4f} seconds')
print('IF-LOF:', evaluation(y_true_shut, y_if_lof_pred_shut),
      f'| Runtime: {time_if_lof_shut:.4f} seconds')

Satellite; (accuracy, F-measure)
IF: (0.7944055944055944, 0.5697560975609756) | Runtime: 0.4829 seconds
LOF: (0.6834498834498834, 0.04321277595115078) | Runtime: 0.8300 seconds
IF-LOF: (0.7651903651903652, 0.5999470479216309) | Runtime: 0.4790 seconds

Shuttle; (accuracy, F-measure)
IF: (0.9496099558017802, 0.7369763980438019) | Runtime: 1.1928 seconds
LOF: (0.90408782614009, 0.061016949152542375) | Runtime: 2.8779 seconds
IF-LOF: (0.8706234596818543, 0.09541441184847622) | Runtime: 4.0874 seconds
